####Higher Order Function
- HOF are functions that operate on complex datatypes such as arrays and maps
- They allow you to pass functions as arguments(such as lambda expressions), apply transformations and return arrays or maps
- They are extremely useful for manipulating arrays without exploding them

#### Commonly used higher order Array function
- TRANSFORM
- FILTER
- EXISTS
- AGGREGATE

####Syntax
------------------------------------------------------------------------------------
`<function_name> (<column_name>, <lambda_expression>)`
- lambda_expression: `element -> expression`

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW order_items
AS
SELECT 
    *
FROM VALUES
    (1, array('smartphone','laptop','monitor')),
    (2, array('tablet','headphones','smartwatch')),
    (3, array('keyboard','mouse','charger'))
AS orders(order_id, items)

####1. TRANSFORM

In [0]:
%sql
SELECT
    TRANSFORM(
        items,
        x -> UPPER(x)
    )
FROM order_items

####2. Filter

In [0]:
%sql
SELECT
    FILTER(
        items,
        x -> x LIKE '%smart%'
    )
FROM order_items

####3. Exists 

In [0]:
%sql
SELECT
    EXISTS(
        items,
        x -> x== 'monitor'
    )
FROM order_items 


#### Working with struct array

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW order_items1
AS
SELECT 
    *
FROM VALUES
    (1, array(
        named_struct('name','smartphone','price',699),
        named_struct('name','laptop','price',1199),
        named_struct('name','monitor','price',399)
    )),
    (2, array(
        named_struct('name','tablet','price',599),
        named_struct('name','headphones','price',199),
        named_struct('name','smartwatch','price',299)
    )),
    (3, array(
        named_struct('name','keyboard','price',89),
        named_struct('name','mouse','price',59)
    ))
AS orders(order_id, items)
    

In [0]:
%sql
SELECT * FROM order_items1

#### 1. Convert all item name to upper case amd add 10% tax on each item

In [0]:
%sql
SELECT
    TRANSFORM(
        items,
        x -> UPPER(x.name)
    ),
    TRANSFORM(
        items,
        x -> x.price * 1.1
    )
FROM order_items1

In [0]:
%sql
SELECT
    TRANSFORM(
        items,
        x -> named_struct('name', UPPER(x.name),
                            'price', ROUND(x.price * 1.1, 2)
        )) AS items_with_tax
FROM order_items1

In [0]:
%sql
SELECT
    AGGREGATE(
        items, 0, (acc,x) -> acc + x.price
    ) AS total_order_price
FROM order_items1

### MAP Functions
A map is a collection of key-value pairs, like a dictionary
`{'laptop': 1200, 'phone':699}`

#### Commonly used higher oder map functions
- Transform_values
- Transform_keys
- MAP_FILTER

#### Syntax
-----
< function_name > (map_column, lambda_expression)

lambda_expression: (key, value) -> expression


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW order_item_prices AS
SELECT * FROM VALUES
    (1, MAP('headphones', 199, 'smartwatch', 299)),
    (2, MAP('smartphone', 699, 'laptop', 1199, 'monitor', 299)),
    (3, MAP('tablet', 599, 'mouse', 59, 'keyboard', 89))
    AS orders(order_id, item_prices)

In [0]:
%sql
SELECT * FROM order_item_prices

#### 1. Convert all item names to be UPPERCASE (transform_keys function)

In [0]:
%sql
SELECT 
    TRANSFORM_KEYS(
        item_prices, (item, price) -> UPPER(item)
    ) AS item_prices
FROM order_item_prices

####2. Apply 10% tax on the price (transform_values function)

In [0]:
%sql
SELECT 
    TRANSFORM_VALUES(
        item_prices, (item, price) -> round(price * 1.1,2)
    ) AS item_prices_with_tax
FROM order_item_prices

3. Filter items only with price above $500 (using MAP_FILTER)

In [0]:
%sql
SELECT 
    MAP_FILTER(
        item_prices, (item, price) -> round(price * 1.1,2) > 500
    ) AS item_prices_with_tax
FROM order_item_prices